Example notebook `python/Notebooks/snippets/solving.ipynb`: the examples of the user manual on simulation settings,
output, the renderer around the solver, and the errors of a solve (the pages *Basics* and *Performance and errors*);
each example is tagged `part-<name>` and shown where the manual explains it.

In [1]:
import exudyn
import exudyn as exu
from exudyn.utilities import *

SC = exu.SystemContainer()
mbs = SC.AddSystem()

## Simulation settings

In [2]:
simulationSettings = exu.SimulationSettings()

tEnd = 10 #10 seconds of simulation time:
h = 0.01  #step size (gives 1000 steps)
simulationSettings.timeIntegration.endTime = tEnd
#steps for time integration must be integer:
simulationSettings.timeIntegration.numberOfSteps = int(tEnd/h)
#assigns a new tolerance for Newton's method:
simulationSettings.timeIntegration.newton.relativeTolerance = 1e-9
#write some output while the solver is active (SLOWER):
simulationSettings.timeIntegration.verboseMode = 2
#write solution every 0.1 seconds:
simulationSettings.solution.file.writePeriod = 0.1
#use sparse matrix storage and solver (package Eigen):
simulationSettings.linearSolver.solverType = exu.LinearSolverType.EigenSparse

## Output

In [3]:
#create a new simulationSettings structure:
simulationSettings = exu.SimulationSettings()

#activate writing to solution file:
simulationSettings.solution.file.write = True
#write results every 1ms:
simulationSettings.solution.file.writePeriod = 0.001

#assign new filename to solution file
simulationSettings.solution.file.name= "myOutput.txt"

#do not export certain coordinates:
simulationSettings.solution.file.export.dataCoordinates = False

## The renderer around the solver

A mass point on a spring, with a sensor, which the following examples solve:

In [4]:
oGround = mbs.CreateGround()
oMass = mbs.CreateMassPoint(referencePosition=[1,0,0], mass=1)
oSpring = mbs.CreateSpringDamper(bodyNumbers=[oGround, oMass], stiffness=100, damping=1)
sensorNumber = mbs.AddSensor(SensorBody(bodyNumber=oMass, storeInternal=True,
                                        outputVariableType=exu.OutputVariableType.Position))
mbs.Assemble()
simulationSettings = exu.SimulationSettings()
simulationSettings.timeIntegration.numberOfSteps = 100
simulationSettings.timeIntegration.endTime = 1

In [5]:
SC.renderer.Start()               #open the window
mbs.SolveDynamic(simulationSettings)
SC.renderer.DoIdleTasks()         #wait for a key press, so the window stays
SC.renderer.Stop()                #close it

## Catching a solver failure

In [6]:
try:
    mbs.SolveDynamic(simulationSettings)
except exudyn.SolverError:
    #the solver stopped: retry with smaller steps
    simulationSettings.timeIntegration.numberOfSteps *= 10
    mbs.SolveDynamic(simulationSettings)
except exudyn.ModelError as e:
    #the model itself is wrong - retrying will not help
    print('this model cannot be solved:', e)

In a parameter variation:

In [7]:
def ParameterFunction(parameterDict):
    ...
    try: mbs.SolveDynamic(simulationSettings)
    except exudyn.ExudynError: return 1e10   #a very bad score, so the optimizer moves away from here
    return mbs.GetSensorValues(sensorNumber)[0]

print('score:', ParameterFunction({}))

score: 1.0


## An error inside a user function

The spring force as a user function that divides by zero:

In [8]:
def springForceUserFunction(mbs, t, itemNumber, deltaL, deltaL_t, stiffness, damping, force):
    return stiffness*deltaL/0.
mbs.SetObjectParameter(oSpring, 'springForceUserFunction', springForceUserFunction)
mbs.Assemble()

In [9]:
try:
    mbs.SolveDynamic(simulationSettings)
except exudyn.ModelError as e:
    print(type(e.__cause__))   #<class 'ZeroDivisionError'>, raised in your user function


******************************
DYNAMIC SOLVER FAILED:
  use showHints=True to show helpful information
******************************

<class 'ZeroDivisionError'>
